# Candidate Recommendation Dataset Pipeline (v2)
## Fix Hard-Negative Collapse, Distance-Weighted Dissimilar Sampling, & Full Gold/Silver/Bronze Construction

This notebook implements the updated candidate recommendation pipeline (`anchor_item`, `anchor_category`, `candidate_item`, `candidate_category`, `label`, `sample_type`, `similarity_score`, `same_category`).

### Key Enhancements in v2:
1. **Fix Hard-Negative Collapse**: Sources hard negatives from the **top-5** most dissimilar categories per anchor category using **distance-weighted random choice** (probabilities proportional to cosine dissimilarity).
2. **Similarity Safety Floor Filter**: Applies a safety floor (`0.55`) to reject hard negative candidate draws that happen to be semantically close to the anchor item.
3. **Same-Category Indicator (`same_category`)**: Adds a boolean column `same_category` (`candidate_category == anchor_category`) to flag substitute vs. complementary pairings.
4. **Full Gold/Silver Positives & Bronze Soft Negatives**: Includes ALL available items from `gold_items` (`positive_gold`) and `silver_items` (`positive_silver`), plus 3 cross-category items (`positive_cross_category`). Includes ALL available items from `bronze_items` (`soft_negative`), topped up with `hard_negative` candidates to maintain an exact **1:2 positive-to-negative sampling ratio**.
5. **Conflict Resolution**: Enforces positive labels (`label = 1`) for any candidate appearing as both positive and negative for an anchor.


## Stage 0: Configuration & Hyperparameters

Centralized configuration for embedding models, top-K dissimilarity, distance weighting, safety floor, cluster bounds, and random seeds.


In [1]:
import os
import ast
import json
import random
import time
import pickle
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering, KMeans
from sentence_transformers import SentenceTransformer

# Configuration Parameters
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
K_SIMILAR = 1               # Top-1 similar category for cross-category positives
TOP_K_DISSIMILAR = 5        # Top-5 dissimilar categories for distance-weighted hard negatives
N_CROSS_POSITIVE = 3        # Cross-category positives per anchor
NEG_TO_POS_RATIO = 2.0      # Exactly 2 negatives per positive sample (1:2 ratio)
SIMILARITY_SAFETY_FLOOR = 0.55 # Reject hard negatives with cosine similarity > 0.55
ITEMS_PER_CLUSTER = 15      # Target item count per sub-cluster in Stage 2
MIN_CLUSTERS = 2            # Minimum sub-clusters per category
MAX_CLUSTERS = 10           # Maximum sub-clusters per category
MIN_ITEMS_FOR_CLUSTERING = 4 # Fall back to 1 cluster if category has < 4 items
RANDOM_SEED = 42

INPUT_FILE = "final_pair_data.csv"
OUTPUT_FILE_V2 = "triplet_dataset_v2.csv"
OUTPUT_FILE_MAIN = "triplet_dataset.csv"
CACHE_EMBEDDINGS_FILE = "item_embeddings_50k.pkl"

def set_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)

set_seeds()
print("Configuration v2 loaded successfully. Random seeds initialized.")


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


W0816 15:13:44.067000 792 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Configuration v2 loaded successfully. Random seeds initialized.


## Stage 1: Load & Parse Data

We load `final_pair_data.csv` and parse `gold_items`, `silver_items`, and `bronze_items` strings via `ast.literal_eval`.


In [2]:
def parse_list(val):
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        try:
            return json.loads(val)
        except Exception:
            return []

print(f"Loading dataset from {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} source rows.")

source_item_col = 'item' if 'item' in df.columns else 'source_item'
source_cat_col = 'category' if 'category' in df.columns else 'source_category'

df['gold_parsed'] = df['gold_items'].apply(parse_list)
df['silver_parsed'] = df['silver_items'].apply(parse_list)
df['bronze_parsed'] = df['bronze_items'].apply(parse_list)

print("Parsed gold_items, silver_items, and bronze_items successfully.")
df[[source_item_col, source_cat_col, 'gold_parsed', 'silver_parsed', 'bronze_parsed']].head(3)


Loading dataset from final_pair_data.csv...


Loaded 49990 source rows.


Parsed gold_items, silver_items, and bronze_items successfully.


,item,category,gold_parsed,silver_parsed,bronze_parsed
0,Amul Milk,Dairy & Bread,"[Modern Family Pack White Bread, Britannia San...","[Gowardhan Classic Edition Unsalted Butter, Mo...","[Country Delight Cubic Cut Paneer, Freshly Fre..."
1,Mother Dairy Toned Milk,Dairy & Bread,"[Harvest Gold Oat Bread, Fresho Whole Wheat Br...",[Kitchens of India Export Quality Paneer Butte...,"[Freshly Gold Reserve Cubic Cut Paneer, Herita..."
2,Mother Dairy Milk,Dairy & Bread,"[Britannia Whole Wheat Bread, Bonn Whole Wheat...",[Kohinoor Masala Combo - Chicken 15 Gm + Meat ...,"[Country Delight Soft Paneer, Cavins Super Val..."


## Stage 2: Category Identity & BGE Item Embeddings

Category memberships are established strictly from `source_item` and `source_category`. Unique item BGE embeddings (`BAAI/bge-small-en-v1.5`) are loaded or generated. Each category mean vector is computed and normalized.


In [3]:
cat_to_items = df.groupby(source_cat_col)[source_item_col].unique().to_dict()
all_unique_items = sorted(list(df[source_item_col].unique()))
item_to_cat = dict(zip(df[source_item_col], df[source_cat_col]))

print(f"Total unique source items: {len(all_unique_items)}")
print(f"Total unique categories: {len(cat_to_items)}")

if os.path.exists(CACHE_EMBEDDINGS_FILE):
    print(f"Loading cached BGE embeddings from {CACHE_EMBEDDINGS_FILE}...")
    with open(CACHE_EMBEDDINGS_FILE, 'rb') as f:
        cache = pickle.load(f)
    item_embeddings = cache['embeddings']
    item_to_idx = cache['item_to_idx']
else:
    print(f"Generating embeddings for {len(all_unique_items)} items using {BGE_MODEL_NAME}...")
    model = SentenceTransformer(BGE_MODEL_NAME)
    item_embeddings = model.encode(all_unique_items, normalize_embeddings=True, show_progress_bar=True, batch_size=256)
    item_to_idx = {item: idx for idx, item in enumerate(all_unique_items)}
    with open(CACHE_EMBEDDINGS_FILE, 'wb') as f:
        pickle.dump({'embeddings': item_embeddings, 'item_to_idx': item_to_idx}, f)
    print(f"Saved embeddings cache to {CACHE_EMBEDDINGS_FILE}.")

cat_names = sorted(list(cat_to_items.keys()))
cat_embeddings = {}
for cat in cat_names:
    items = cat_to_items[cat]
    indices = [item_to_idx[it] for it in items]
    mean_vec = item_embeddings[indices].mean(axis=0)
    norm = np.linalg.norm(mean_vec)
    if norm > 0:
        mean_vec = mean_vec / norm
    cat_embeddings[cat] = mean_vec

cat_matrix = np.array([cat_embeddings[cat] for cat in cat_names])
print(f"Computed category mean vectors for {len(cat_names)} categories. Shape: {cat_matrix.shape}")


Total unique source items: 49990
Total unique categories: 17
Loading cached BGE embeddings from item_embeddings_50k.pkl...


Computed category mean vectors for 17 categories. Shape: (17, 384)


## Stage 3: Distance-Weighted Top-5 Dissimilar Category Sampling

To prevent hard-negative category collapse, we select the **top-5** most dissimilar categories per anchor category and compute distance weights $w_i = d_i / \sum d$.


In [4]:
dist_matrix = squareform(pdist(cat_matrix, metric='cosine'))
cat_to_idx = {cat: idx for idx, cat in enumerate(cat_names)}

n_clusters_stage1 = max(2, len(cat_names) // 3)
agg_clustering = AgglomerativeClustering(n_clusters=n_clusters_stage1, metric='cosine', linkage='average')
cluster_labels = agg_clustering.fit_predict(cat_matrix)

print("Exploratory Category Clusters:")
for cat, label in zip(cat_names, cluster_labels):
    print(f"  - Cluster {label}: {cat}")

def most_similar(category, k=K_SIMILAR):
    idx = cat_to_idx[category]
    distances = dist_matrix[idx].copy()
    distances[idx] = np.inf
    top_k_indices = np.argsort(distances)[:k]
    return [cat_names[i] for i in top_k_indices]

def top_k_dissimilar_with_weights(category, k=TOP_K_DISSIMILAR):
    idx = cat_to_idx[category]
    distances = dist_matrix[idx].copy()
    distances[idx] = -np.inf
    top_k_indices = np.argsort(distances)[::-1][:k]
    top_cats = [cat_names[i] for i in top_k_indices]
    top_dists = np.array([distances[i] for i in top_k_indices])
    weights = top_dists / top_dists.sum()
    return top_cats, weights

top_similar_map = {cat: most_similar(cat, k=K_SIMILAR)[0] for cat in cat_names}
top_dissimilar_weighted_map = {cat: top_k_dissimilar_with_weights(cat, k=TOP_K_DISSIMILAR) for cat in cat_names}

print("\nCategory Top-1 Similar & Top-5 Dissimilar Weighted Mapping:")
for cat in cat_names:
    top5_c, top5_w = top_dissimilar_weighted_map[cat]
    w_str = ", ".join([f"{c}({round(w,2)})" for c, w in zip(top5_c, top5_w)])
    print(f"  '{cat}' -> Similar: '{top_similar_map[cat]}' | Top-5 Dissimilar: [{w_str}]")


Exploratory Category Clusters:
  - Cluster 0: Atta/Rice/Dal
  - Cluster 2: Baby Care
  - Cluster 0: Beverages
  - Cluster 0: Breakfast Cereals
  - Cluster 0: Chocolates & Sweets
  - Cluster 0: Dairy & Bread
  - Cluster 3: Fruits & Vegetables
  - Cluster 2: Home Care
  - Cluster 0: Ice Creams & Frozen Desserts
  - Cluster 0: Instant Food
  - Cluster 0: Meat & Seafood
  - Cluster 0: Paan Corner
  - Cluster 2: Personal Care
  - Cluster 4: Pet Care
  - Cluster 2: Pharma & Wellness
  - Cluster 0: Snacks & Biscuits
  - Cluster 1: Stationery & Office

Category Top-1 Similar & Top-5 Dissimilar Weighted Mapping:
  'Atta/Rice/Dal' -> Similar: 'Instant Food' | Top-5 Dissimilar: [Stationery & Office(0.26), Baby Care(0.21), Pet Care(0.19), Ice Creams & Frozen Desserts(0.19), Chocolates & Sweets(0.16)]
  'Baby Care' -> Similar: 'Personal Care' | Top-5 Dissimilar: [Fruits & Vegetables(0.23), Breakfast Cereals(0.2), Paan Corner(0.2), Ice Creams & Frozen Desserts(0.19), Meat & Seafood(0.18)]
  'Beverag

## Stage 4: Stage 2 Clustering (Item-Wise Sub-Clustering)

Category-wise `KMeans` partitions member items into sub-clusters. Candidate searches route to nearest sub-cluster centroids via `nearest_cluster(anchor_embedding, target_category)`.


In [5]:
cat_subclusters = {}

for cat in cat_names:
    items = list(cat_to_items[cat])
    n_items = len(items)
    if n_items < MIN_ITEMS_FOR_CLUSTERING:
        n_c = 1
    else:
        n_c = max(MIN_CLUSTERS, min(MAX_CLUSTERS, int(round(n_items / ITEMS_PER_CLUSTER))))
        n_c = min(n_c, n_items)

    item_vecs = np.array([item_embeddings[item_to_idx[it]] for it in items])
    
    if n_c > 1:
        km = KMeans(n_clusters=n_c, random_state=RANDOM_SEED, n_init=10)
        labels = km.fit_predict(item_vecs)
    else:
        labels = np.zeros(n_items, dtype=int)

    clusters_dict = {}
    for c_id in range(n_c):
        c_mask = (labels == c_id)
        c_items = [items[i] for i in range(n_items) if c_mask[i]]
        c_vecs = item_vecs[c_mask]
        centroid = c_vecs.mean(axis=0)
        norm = np.linalg.norm(centroid)
        if norm > 0:
            centroid = centroid / norm
        clusters_dict[c_id] = {
            "items": c_items,
            "centroid": centroid
        }
    cat_subclusters[cat] = clusters_dict

def nearest_cluster(anchor_embedding, target_category):
    clusters_dict = cat_subclusters[target_category]
    best_cid = None
    best_sim = -np.inf
    for c_id, c_info in clusters_dict.items():
        sim = float(np.dot(anchor_embedding, c_info["centroid"]))
        if sim > best_sim:
            best_sim = sim
            best_cid = c_id
    return best_cid

total_subclusters = sum(len(c) for c in cat_subclusters.values())
print(f"Created {total_subclusters} item-level sub-clusters across {len(cat_names)} categories.")


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Created 157 item-level sub-clusters across 17 categories.


## Stage 5: Persistent PoolCycler Utility

Persistent `PoolCycler` instances hand out items round-robin per `(category, cluster_id)` sub-cluster.


In [6]:
class PoolCycler:
    def __init__(self, items, seed=RANDOM_SEED):
        self.original_items = list(items)
        self.rng = random.Random(seed)
        self.pool = []
        self._reset()

    def _reset(self):
        self.pool = list(self.original_items)
        self.rng.shuffle(self.pool)

    def next(self):
        if not self.original_items:
            return None
        if not self.pool:
            self._reset()
        return self.pool.pop()

pool_cyclers = {}
for cat in cat_names:
    for c_id, c_info in cat_subclusters[cat].items():
        pool_key = (cat, c_id)
        pool_seed = RANDOM_SEED + hash(pool_key) % 100000
        pool_cyclers[pool_key] = PoolCycler(c_info["items"], seed=pool_seed)

print(f"Initialized {len(pool_cyclers)} persistent PoolCycler pools.")


Initialized 157 persistent PoolCycler pools.


## Stage 6: Candidate Dataset Generation & Similarity Safety Floor

Generates candidates per anchor:
- **Positives**: All gold items (`positive_gold`) + all silver items (`positive_silver`) + 3 cross-category items (`positive_cross_category`).
- **Negatives**: Target ratio $2.0 	imes N_{pos}$. Includes all bronze items (`soft_negative`), topped up with `hard_negative` candidates drawn via distance-weighted top-5 dissimilar choice with similarity safety floor filter ($\le 0.55$).
- **Features**: `same_category` boolean flag and BGE cosine similarity score ($ec{v}_{anchor} \cdot ec{v}_{candidate}$).


In [7]:
records = []
rng = random.Random(RANDOM_SEED)

for idx, row in df.iterrows():
    anchor = row[source_item_col]
    anchor_cat = row[source_cat_col]
    anchor_emb = item_embeddings[item_to_idx[anchor]]

    target_sim_cat = top_similar_map[anchor_cat]
    sim_cid = nearest_cluster(anchor_emb, target_sim_cat)
    sim_pool_key = (target_sim_cat, sim_cid)

    top5_dissim_cats, top5_weights = top_dissimilar_weighted_map[anchor_cat]

    positives_for_anchor = []
    pos_seen = set([anchor])

    # 1. Gold Positives
    for g_item in row['gold_parsed']:
        if g_item and g_item not in pos_seen:
            positives_for_anchor.append((g_item, 'positive_gold'))
            pos_seen.add(g_item)

    # 2. Silver Positives
    for s_item in row['silver_parsed']:
        if s_item and s_item not in pos_seen:
            positives_for_anchor.append((s_item, 'positive_silver'))
            pos_seen.add(s_item)

    # 3. Cross-Category Positives
    cross_count = 0
    for _ in range(len(pool_cyclers[sim_pool_key].original_items) + 10):
        if cross_count >= N_CROSS_POSITIVE:
            break
        cand = pool_cyclers[sim_pool_key].next()
        if cand and cand not in pos_seen:
            positives_for_anchor.append((cand, 'positive_cross_category'))
            pos_seen.add(cand)
            cross_count += 1

    # Emit Positive Rows (label = 1)
    for cand_item, stype in positives_for_anchor:
        cand_idx = item_to_idx.get(cand_item)
        if cand_idx is not None:
            sim_score = float(np.dot(anchor_emb, item_embeddings[cand_idx]))
            cand_cat = item_to_cat.get(cand_item, target_sim_cat)
        else:
            sim_score = 0.0
            cand_cat = target_sim_cat

        records.append({
            'anchor_item': anchor,
            'anchor_category': anchor_cat,
            'candidate_item': cand_item,
            'candidate_category': cand_cat,
            'label': 1,
            'sample_type': stype,
            'similarity_score': round(sim_score, 4),
            'same_category': (cand_cat == anchor_cat)
        })

    # Negatives (label = 0)
    total_positives = len(positives_for_anchor)
    target_total_negatives = int(round(total_positives * NEG_TO_POS_RATIO))

    negatives_for_anchor = []
    neg_seen = set(pos_seen)

    # Soft Negatives
    for b_item in row['bronze_parsed']:
        if b_item and b_item not in neg_seen:
            negatives_for_anchor.append((b_item, 'soft_negative'))
            neg_seen.add(b_item)

    # Hard Negatives with Safety Floor
    needed_hard = max(0, target_total_negatives - len(negatives_for_anchor))
    hard_count = 0

    for _ in range(needed_hard * 10 + 50):
        if hard_count >= needed_hard:
            break
        
        chosen_dissim_cat = rng.choices(top5_dissim_cats, weights=top5_weights, k=1)[0]
        dissim_cid = nearest_cluster(anchor_emb, chosen_dissim_cat)
        dissim_pool_key = (chosen_dissim_cat, dissim_cid)

        cand_item = None
        for attempt in range(5):
            candidate_drawn = pool_cyclers[dissim_pool_key].next()
            if candidate_drawn and candidate_drawn not in neg_seen:
                c_idx = item_to_idx.get(candidate_drawn)
                if c_idx is not None:
                    sim_val = float(np.dot(anchor_emb, item_embeddings[c_idx]))
                    if sim_val <= SIMILARITY_SAFETY_FLOOR or attempt == 4:
                        cand_item = candidate_drawn
                        break
                else:
                    cand_item = candidate_drawn
                    break

        if cand_item and cand_item not in neg_seen:
            negatives_for_anchor.append((cand_item, 'hard_negative'))
            neg_seen.add(cand_item)
            hard_count += 1

    # Emit Negative Rows (label = 0)
    for cand_item, stype in negatives_for_anchor:
        cand_idx = item_to_idx.get(cand_item)
        if cand_idx is not None:
            sim_score = float(np.dot(anchor_emb, item_embeddings[cand_idx]))
            cand_cat = item_to_cat.get(cand_item, 'Unknown')
        else:
            sim_score = 0.0
            cand_cat = 'Unknown'

        records.append({
            'anchor_item': anchor,
            'anchor_category': anchor_cat,
            'candidate_item': cand_item,
            'candidate_category': cand_cat,
            'label': 0,
            'sample_type': stype,
            'similarity_score': round(sim_score, 4),
            'same_category': (cand_cat == anchor_cat)
        })

dataset_df = pd.DataFrame(records)
print(f"Generated {len(dataset_df)} total candidate rows.")


Generated 2517486 total candidate rows.


## Stage 7: Conflict Resolution & Quality Verification

Verifies hard-negative category diversity, resolves label conflicts, and confirms zero self-loops or missing categories.


In [8]:
# Conflict Resolution
pos_pairs = set(dataset_df[dataset_df['label'] == 1][['anchor_item', 'candidate_item']].itertuples(index=False, name=None))
conflicting_mask = (dataset_df['label'] == 0) & dataset_df[['anchor_item', 'candidate_item']].apply(tuple, axis=1).isin(pos_pairs)
conflict_count = conflicting_mask.sum()
if conflict_count > 0:
    print(f"Conflict Resolution: Removed {conflict_count} conflicting label=0 rows.")
    dataset_df = dataset_df[~conflicting_mask].reset_index(drop=True)

dataset_df = dataset_df.drop_duplicates(subset=['anchor_item', 'candidate_item'], keep='first').reset_index(drop=True)

self_loops = (dataset_df['anchor_item'] == dataset_df['candidate_item']).sum()
null_cand_cats = dataset_df['candidate_category'].isna().sum()

pos_df = dataset_df[dataset_df['label'] == 1]
neg_df = dataset_df[dataset_df['label'] == 0]

print("==================================================")
print("             DATASET QUALITY REPORT (v2)          ")
print("==================================================")
print(f"1. Total positive rows (label=1) : {len(pos_df)}")
print(f"2. Total negative rows (label=0) : {len(neg_df)}")
print(f"3. Overall Neg:Pos sampling ratio: {round(len(neg_df) / max(1, len(pos_df)), 2)} (Target: ~2.0)")
print(f"4. Anchor == Candidate count     : {self_loops} (Target: 0)")
print(f"5. Null candidate categories     : {null_cand_cats} (Target: 0)")
print("\n6. Unique candidate items & categories per sample_type:")
sample_type_stats = dataset_df.groupby('sample_type').agg(
    total_rows=('candidate_item', 'count'),
    unique_candidate_items=('candidate_item', 'nunique'),
    unique_candidate_categories=('candidate_category', 'nunique')
)
print(sample_type_stats)

print("\n7. Hard Negative Target Category Distribution:")
print(dataset_df[dataset_df['sample_type'] == 'hard_negative']['candidate_category'].value_counts())

print("\n8. Breakdown of same_category flag per sample_type:")
print(dataset_df.groupby(['sample_type', 'same_category']).size().unstack(fill_value=0))
print("==================================================")

assert self_loops == 0, "Sanity check failed: anchor_item == candidate_item found!"
assert null_cand_cats == 0, "Sanity check failed: null candidate_category found!"
print("ALL QUALITY SANITY CHECKS PASSED SUCCESSFULLY!")


             DATASET QUALITY REPORT (v2)          
1. Total positive rows (label=1) : 839162
2. Total negative rows (label=0) : 1678324
3. Overall Neg:Pos sampling ratio: 2.0 (Target: ~2.0)
4. Anchor == Candidate count     : 0 (Target: 0)
5. Null candidate categories     : 0 (Target: 0)

6. Unique candidate items & categories per sample_type:


                         total_rows  unique_candidate_items  \
sample_type                                                   
hard_negative               1398878                   16132   
positive_cross_category      149970                   21701   
positive_gold                392004                   31596   
positive_silver              297188                   26602   
soft_negative                279446                   36193   

                         unique_candidate_categories  
sample_type                                           
hard_negative                                     12  
positive_cross_category                            8  
positive_gold                                     17  
positive_silver                                   17  
soft_negative                                     17  

7. Hard Negative Target Category Distribution:


candidate_category
Fruits & Vegetables             206639
Ice Creams & Frozen Desserts    197682
Stationery & Office             191246
Breakfast Cereals               155616
Baby Care                       131284
Chocolates & Sweets             122841
Pet Care                        113726
Meat & Seafood                  102455
Paan Corner                      84757
Personal Care                    36330
Pharma & Wellness                32424
Home Care                        23878
Name: count, dtype: int64

8. Breakdown of same_category flag per sample_type:


same_category              False   True 
sample_type                             
hard_negative            1398878       0
positive_cross_category   149970       0
positive_gold             113938  278066
positive_silver           116890  180298
soft_negative             103445  176001
ALL QUALITY SANITY CHECKS PASSED SUCCESSFULLY!


## Stage 8: Save Dataset

Exporting candidate dataset to `triplet_dataset_v2.csv` and `triplet_dataset.csv`.


In [9]:
out_cols = ['anchor_item', 'anchor_category', 'candidate_item', 'candidate_category', 'label', 'sample_type', 'similarity_score', 'same_category']
dataset_df[out_cols].to_csv(OUTPUT_FILE_V2, index=False)
dataset_df[out_cols].to_csv(OUTPUT_FILE_MAIN, index=False)
print(f"Successfully saved candidate dataset to '{OUTPUT_FILE_V2}' and '{OUTPUT_FILE_MAIN}'.")
print("\nFirst 10 rows preview:")
pd.read_csv(OUTPUT_FILE_V2).head(10)


Successfully saved candidate dataset to 'triplet_dataset_v2.csv' and 'triplet_dataset.csv'.

First 10 rows preview:


,anchor_item,anchor_category,candidate_item,candidate_category,label,sample_type,similarity_score,same_category
0,Amul Milk,Dairy & Bread,Modern Family Pack White Bread,Dairy & Bread,1,positive_gold,0.5851,True
1,Amul Milk,Dairy & Bread,Britannia Sandwich Bread,Dairy & Bread,1,positive_gold,0.5444,True
2,Amul Milk,Dairy & Bread,Tata Q Select Choice Pav Bhaji Ready to Eat,Instant Food,1,positive_gold,0.5785,False
3,Amul Milk,Dairy & Bread,TGL Co. Happy Belly Soothing Herbal Tea for Belly,Beverages,1,positive_gold,0.5705,False
4,Amul Milk,Dairy & Bread,Twinings Value Pack Earl Grey Tea,Beverages,1,positive_gold,0.5054,False
5,Amul Milk,Dairy & Bread,PRISTINE Deccan Gold - Premium Dust Tea,Beverages,1,positive_gold,0.5599,False
6,Amul Milk,Dairy & Bread,Sleepy Owl Gold Reserve Espresso Coffee,Beverages,1,positive_gold,0.5659,False
7,Amul Milk,Dairy & Bread,Lavazza Ground Coffee - Qualita Rossa,Beverages,1,positive_gold,0.5524,False
8,Amul Milk,Dairy & Bread,Nescafe Super Value Filter Coffee,Beverages,1,positive_gold,0.5463,False
9,Amul Milk,Dairy & Bread,Saffola Steel Cut Oats,Breakfast Cereals,1,positive_gold,0.5187,False
